# Car AI Project - Preprocessing Decision Tree

## 1. Libraries

In [0]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import KBinsDiscretizer


import joblib

In [0]:
# Import project configuration
import sys
import os

# Add parent directory to path to import config
sys.path.append("..")
from config import *

print("PROJECT CONFIGURATION LOADED")
print(f"\nBASE_PATH: {BASE_PATH}")
print(f"\nData Paths:")
print(f"   - RAW_DATA_PATH: {RAW_DATA_PATH}")
print(f"   - FEATURES_PATH: {FEATURES_PATH}")
print(f"   - PROCESSED_DATA_PATH: {PROCESSED_DATA_PATH}")
print(f"   - TRAIN_TEST_PATH: {TRAIN_TEST_PATH}")
print(f"   - METRICS_PATH: {METRICS_PATH}")
print(f"\nModel Path:")
print(f"   - MODEL_PATH: {MODEL_PATH}")
print(f"\nUnity Catalog:")
print(f"   - SOURCE_CSV_FILE: {SOURCE_CSV_FILE}")
print(f"   - RAW_CARS_TABLE: {RAW_CARS_TABLE}")
print(f"   - CLEANED_CARS_TABLE: {CLEANED_CARS_TABLE}")

PROJECT CONFIGURATION LOADED

BASE_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject

Data Paths:
   - RAW_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/raw
   - FEATURES_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features
   - PROCESSED_DATA_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/processed
   - TRAIN_TEST_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/train_test
   - METRICS_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/metrics

Model Path:
   - MODEL_PATH: /Workspace/Users/maria.petralia@gmail.com/CarAIProject/models

Unity Catalog:
   - SOURCE_CSV_FILE: /Volumes/workspace/caraiproject/caraiproject/Cars_Datasets_2025.csv
   - RAW_CARS_TABLE: workspace.caraiproject.raw_cars_data_gathered
   - CLEANED_CARS_TABLE: workspace.caraiproject.cleaned_cars_data


## 2. Load the DT Dataset

In [0]:
# Import unified engineered dataset from Delta Table
# This is the SAME table used by 05_1_Preprocessing_LinearRegression
# Ensures fair comparison and reproducible train/test splits

source_table = "workspace.caraiproject.engineered_cars_data"

# Read Delta Table using Spark (required for Delta format)
spark_df = spark.table(source_table)
row_count_spark = spark_df.count()

# Convert to pandas DataFrame for data manipulation
df = spark_df.toPandas()

# Display dataset information
print("UNIFIED ENGINEERED DATASET")
print(f"\nShape: {df.shape}")

print(f"\nColumns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:30s} ({df[col].dtype})")

display(df.head())

print(f"\nDataset ready for Decision Tree Preprocessing")

UNIFIED ENGINEERED DATASET

Shape: (1209, 32)

Columns (32):
   1. make                           (object)
   2. model                          (object)
   3. engine_displacement_cc         (float64)
   4. battery_capacity_kwh           (float64)
   5. acc_0_100_min                  (float64)
   6. acc_0_100_max                  (float64)
   7. acc_0_100_mean                 (float64)
   8. acc_0_100_is_range             (int64)
   9. acc_0_100_is_instant           (int64)
  10. is_commercial                  (int64)
  11. horsepower_min                 (float64)
  12. horsepower_max                 (float64)
  13. horsepower_mean                (float64)
  14. horsepower_is_range            (int64)
  15. top_speed_kmh                  (float64)
  16. seats                          (float64)
  17. torque_nm                      (int64)
  18. price_usd                      (float64)
  19. is_ev                          (int64)
  20. is_hybrid                      (int64)
  21. is_phev  

make,model,engine_displacement_cc,battery_capacity_kwh,acc_0_100_min,acc_0_100_max,acc_0_100_mean,acc_0_100_is_range,acc_0_100_is_instant,is_commercial,horsepower_min,horsepower_max,horsepower_mean,horsepower_is_range,top_speed_kmh,seats,torque_nm,price_usd,is_ev,is_hybrid,is_phev,fuel_macro,segment,is_luxury_brand,is_performance_luxury_brand,is_premium_brand,acceleration_class,hp_class,log_price,performance_score,acc_missing_flag,fe_timestamp
Ferrari,SF90 STRADALE,3990.0,null,2.5,2.5,2.5,0,0,0,963.0,963.0,963.0,0,340.0,2.0,800,1100000.0,0,1,1,Hybrid,Supercar,0,1,0,Extreme,Extreme,13.910821646859095,105.2,0,2026-05-11T06:59:11.631Z
Rolls-Royce,PHANTOM,6749.0,null,5.3,5.3,5.3,0,0,0,563.0,563.0,563.0,0,250.0,5.0,900,460000.0,0,0,0,Petrol,Sportscar,1,0,0,Medium,High,13.038983942375959,62.5,0,2026-05-11T06:59:11.631Z
Mercedes-Benz,GT 63 S,3982.0,null,3.2,3.2,3.2,0,0,0,630.0,630.0,630.0,0,250.0,4.0,900,161000.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,11.989165855127435,71.3,0,2026-05-11T06:59:11.631Z
Audi,AUDI R8 Gt,5204.0,null,3.6,3.6,3.6,0,0,0,602.0,602.0,602.0,0,320.0,2.0,560,253290.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,12.442294304367605,65.4,0,2026-05-11T06:59:11.631Z
BMW,Mclaren 720s,3994.0,null,2.9,2.9,2.9,0,0,0,710.0,710.0,710.0,0,341.0,2.0,770,499000.0,0,0,0,Petrol,Supercar,0,0,1,Extreme,Extreme,13.120363378739663,79.21,0,2026-05-11T06:59:11.631Z



Dataset ready for Decision Tree Preprocessing


In [0]:
# drop the cleaning_timestamp column and create a new dataframe
df_dt = df.drop(columns=["fe_timestamp"])

In [0]:
df_dt

,make,model,engine_displacement_cc,battery_capacity_kwh,acc_0_100_min,acc_0_100_max,acc_0_100_mean,acc_0_100_is_range,acc_0_100_is_instant,is_commercial,horsepower_min,horsepower_max,horsepower_mean,horsepower_is_range,top_speed_kmh,seats,torque_nm,price_usd,is_ev,is_hybrid,is_phev,fuel_macro,segment,is_luxury_brand,is_performance_luxury_brand,is_premium_brand,acceleration_class,hp_class,log_price,performance_score,acc_missing_flag
0,Ferrari,SF90 STRADALE,3990.0,NaN,2.5,2.5,2.5,0,0,0,963.0,963.0,963.0,0,340.0,2.0,800,1100000.0,0,1,1,Hybrid,Supercar,0,1,0,Extreme,Extreme,13.910822,105.20,0
1,Rolls-Royce,PHANTOM,6749.0,NaN,5.3,5.3,5.3,0,0,0,563.0,563.0,563.0,0,250.0,5.0,900,460000.0,0,0,0,Petrol,Sportscar,1,0,0,Medium,High,13.038984,62.50,0
2,Mercedes-Benz,GT 63 S,3982.0,NaN,3.2,3.2,3.2,0,0,0,630.0,630.0,630.0,0,250.0,4.0,900,161000.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,11.989166,71.30,0
3,Audi,AUDI R8 Gt,5204.0,NaN,3.6,3.6,3.6,0,0,0,602.0,602.0,602.0,0,320.0,2.0,560,253290.0,0,0,0,Petrol,Supercar,0,0,1,Fast,Extreme,12.442294,65.40,0
4,BMW,Mclaren 720s,3994.0,NaN,2.9,2.9,2.9,0,0,0,710.0,710.0,710.0,0,341.0,2.0,770,499000.0,0,0,0,Petrol,Supercar,0,0,1,Extreme,Extreme,13.120363,79.21,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1204,Toyota,Crown Signia,2487.0,NaN,7.6,7.6,7.6,0,0,0,240.0,240.0,240.0,0,180.0,5.0,239,43590.0,1,1,0,Electric,SUV,0,0,0,Medium,Medium,10.682606,20.59,0
1205,Toyota,4Runner (6th Gen),2393.0,NaN,6.8,6.8,6.8,0,0,0,326.0,326.0,326.0,0,180.0,7.0,630,50000.0,0,1,0,Hybrid,Sportscar,0,0,0,Medium,High,10.819798,33.90,0
1206,Toyota,Corolla Cross,NaN,NaN,8.0,8.0,8.0,0,0,0,169.0,196.0,182.5,1,190.0,5.0,190,25210.0,0,1,0,Hybrid,SUV,0,0,0,Slow,Medium,10.135036,14.05,0
1207,Toyota,C-HR+,NaN,NaN,7.9,7.9,7.9,0,0,0,140.0,198.0,169.0,1,180.0,5.0,190,33000.0,0,1,0,Hybrid,SUV,0,0,0,Medium,Medium,10.404293,12.70,0


## 3. Preprocessing

**Unified Base + Tree-Based Model Advantages**

This preprocessing notebook starts from the **same engineered dataset** used by Linear Regression (05_1).  
All models see the **same features initially**, ensuring fair comparison.

**Tree-Based Model Advantages:**

1. **Keep structural features** (`engine_displacement_cc`, `battery_capacity_kwh`)  
   - Tree models can naturally handle structural missingness via splits  
   - These features provide valuable information for non-linear interactions  
   - Imputation strategy: `0` (structural null → explicit zero encoding)

2. **No scaling required**  
   - Tree-based splits are invariant to feature scale  
   - Saves computation and preserves interpretability

3. **Handle non-linearity natively**  
   - No need for polynomial terms or manual interaction features  
   - Trees discover complex patterns automatically

All engineered features from 04_Feature_Engineering are retained, including:  
`performance_score` (composite performance indicator)  
`acc_missing_flag` (structural missingness indicator)  
Brand tiers, segments, and all derived features

In [0]:
# Numerical features for tree-based models
# Includes all numerical features from unified engineered dataset
numerical_features_dt = [
    "acc_0_100_min",
    "acc_0_100_max",
    "acc_0_100_mean",
    "acc_0_100_is_range",
    "acc_0_100_is_instant",

    "horsepower_min",
    "horsepower_max",
    "horsepower_mean",
    "horsepower_is_range",

    "top_speed_kmh",
    "torque_nm",
    "seats",
    
    "performance_score",      # Composite performance indicator (from unified base)
    "acc_missing_flag"         # Structural missingness flag (from unified base)
]

In [0]:
structural_features_dt = [
    "engine_displacement_cc",
    "battery_capacity_kwh"
]

In [0]:
categorical_features_dt = [
    "make",
    "fuel_macro",
    "acceleration_class",
    "hp_class",
    "segment"
]

In [0]:
binary_features_dt = [
    "is_ev",
    "is_hybrid",
    "is_phev",
    "is_commercial",
    "is_luxury_brand",
    "is_premium_brand",
    "is_performance_luxury_brand"
]

In [0]:
target = 'log_price'

In [0]:
# Create price bins for stratified train/test split
# Ensures proportional representation of all price ranges (including supercars) in train and test
# Using price_usd for bins: more interpretable than log_price (reflects actual market segments)
price_bins = pd.qcut(df_dt['price_usd'], q=10, labels=False, duplicates='drop')

print(f"Price bins created: {price_bins.nunique()} bins")
print(f"Samples per bin:")
print(price_bins.value_counts().sort_index())

Price bins created: 10 bins
Samples per bin:
price_usd
0    128
1    141
2    124
3     96
4    117
5    130
6    110
7    121
8    121
9    121
Name: count, dtype: int64


### 3.1 Split Train/Test

#### Train/Test Split with Log Target

In [0]:
# Define feature matrix X by dropping target columns
X = df_dt.drop(columns=["price_usd", "log_price"])

# Define target vector y
y = df_dt[target]

# Split data into training and test sets with stratification
# Stratification ensures proportional representation of all price ranges
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X, y, test_size=0.2, random_state=3, stratify=price_bins
)

print(f"✅ Stratified train/test split completed (log_price target)")
print(f"   X_train: {X_train_dt.shape}")
print(f"   X_test: {X_test_dt.shape}")
print(f"   Stratified by: price_usd bins (ensures all price ranges represented)")

✅ Stratified train/test split completed (log_price target)
   X_train: (967, 29)
   X_test: (242, 29)
   Stratified by: price_usd bins (ensures all price ranges represented)


#### Preprocessing Pipelines and ColumnTransformer Setup

In [0]:
# Pipeline for numerical features: imputes missing values with the median
numerical_pipeline_dt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Pipeline for categorical features: imputes missing values with the most frequent value, then applies one-hot encoding
categorical_pipeline_dt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Pipeline for structural features: imputes missing values with 0
structural_pipeline_dt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

# ColumnTransformer applies the appropriate pipeline to each feature type
preprocess_dt = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline_dt, numerical_features_dt),
        ('cat', categorical_pipeline_dt, categorical_features_dt),
        ('bin', 'passthrough', binary_features_dt),
        ('struct', structural_pipeline_dt, structural_features_dt)
    ]
)

#### Saving Train/Test Splits and Preprocessing Objects

In [0]:
# Save train/test splits
joblib.dump(X_train_dt, os.path.join(TRAIN_TEST_PATH, "X_train_dt.pkl"))
joblib.dump(X_test_dt, os.path.join(TRAIN_TEST_PATH, "X_test_dt.pkl"))
joblib.dump(y_train_dt, os.path.join(TRAIN_TEST_PATH, "y_train_dt.pkl"))
joblib.dump(y_test_dt, os.path.join(TRAIN_TEST_PATH, "y_test_dt.pkl"))

# Save preprocessing pipeline
joblib.dump(preprocess_dt, os.path.join(TRAIN_TEST_PATH, "preprocess_dt.pkl"))

# Save feature lists
joblib.dump(numerical_features_dt, os.path.join(FEATURES_PATH, "numerical_features_dt.pkl"))
joblib.dump(categorical_features_dt, os.path.join(FEATURES_PATH, "categorical_features_dt.pkl"))
joblib.dump(binary_features_dt, os.path.join(FEATURES_PATH, "binary_features_dt.pkl"))
joblib.dump(structural_features_dt, os.path.join(FEATURES_PATH, "structural_features_dt.pkl"))

['/Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features/structural_features_dt.pkl']

#### Train/Test Split with Original Target (price_usd) and Saving Objects

In [0]:
# Set the new target
target_usd = 'price_usd'

# Create X and y for price_usd
X_usd = df_dt.drop(columns=["price_usd", "log_price"])
y_usd = df_dt[target_usd]

# Split train/test using THE SAME stratification bins
X_train_usd_df, X_test_usd_df, y_train_usd_df, y_test_usd_df = train_test_split(
    X_usd, y_usd, test_size=0.2, random_state=3, stratify=price_bins
)

print(f"Stratified train/test split completed (price_usd target)")
print(f"   X_train: {X_train_usd_df.shape}")
print(f"   X_test: {X_test_usd_df.shape}")
print(f"   Same rows as log_price split (only target differs)")

# Save USD splits
joblib.dump(X_train_usd_df, os.path.join(TRAIN_TEST_PATH, "X_train_usd_df.pkl"))
joblib.dump(X_test_usd_df, os.path.join(TRAIN_TEST_PATH, "X_test_usd_df.pkl"))
joblib.dump(y_train_usd_df, os.path.join(TRAIN_TEST_PATH, "y_train_usd_df.pkl"))
joblib.dump(y_test_usd_df, os.path.join(TRAIN_TEST_PATH, "y_test_usd_df.pkl"))

# Save USD preprocessing pipeline
joblib.dump(preprocess_dt, os.path.join(TRAIN_TEST_PATH, "preprocess_usd_dt.pkl"))

# Save USD feature lists
joblib.dump(numerical_features_dt, os.path.join(FEATURES_PATH, "numerical_features_usd_dt.pkl"))
joblib.dump(categorical_features_dt, os.path.join(FEATURES_PATH, "categorical_features_usd_dt.pkl"))
joblib.dump(binary_features_dt, os.path.join(FEATURES_PATH, "binary_features_usd_dt.pkl"))
joblib.dump(structural_features_dt, os.path.join(FEATURES_PATH, "structural_features_usd_dt.pkl"))

Stratified train/test split completed (price_usd target)
   X_train: (967, 29)
   X_test: (242, 29)
   Same rows as log_price split (only target differs)


['/Workspace/Users/maria.petralia@gmail.com/CarAIProject/data/features/structural_features_usd_dt.pkl']